In [1]:
from langchain_community.document_loaders import JSONLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import Chroma

import shutil
import os

### **Creating a vector base**

This code loads data from JSON file, where scrapped data is stored. After that the data is chunked, embedded and saved in a file

**Loading data from directory**

In [2]:
print("Loading data from JSON file...")
def metadata_func(record: dict, metadata: dict) -> dict:
    metadata["title"] = record.get("title")
    metadata["category"] = record.get("source_category")
    return metadata

loader = JSONLoader(
    file_path='./wiki_data.jsonl',
    jq_schema='.',
    content_key="content",
    json_lines=True,
    metadata_func=metadata_func
)

documents = loader.load()

Loading data from JSON file...


**Chunking articles**

In [3]:
print("Splitting text into sections...")
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1200,
    chunk_overlap=50,
    length_function=len
)
text_pieces = text_splitter.split_documents(documents)

Splitting text into sections...


**Embedding**

In [4]:
print("Loading embedding model...")
embeddings = HuggingFaceEmbeddings(model_name='sentence-transformers/all-MiniLM-L6-v2')

Loading embedding model...


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


**Saving into a file**

In [5]:
persist_dir = "./chroma_terraria_db"

if os.path.exists(persist_dir):
    print(f"Deleting old database from {persist_dir}...")
    shutil.rmtree(persist_dir)

Deleting old database from ./chroma_terraria_db...


In [6]:
print("Creating and saving vector base...")
vector_base = Chroma.from_documents(
    documents=text_pieces,
    embedding=embeddings,
    persist_directory="./chroma_terraria_db"
)
print("Vector base was successfully saved")

Creating and saving vector base...
Vector base was successfully saved
